# Custom Agent

### It is a  simple agent that adds a retry layer on top of a RouterQueryEngine, allowing it to retry queries until the task is complete. We build this on top of both a SQL tool and a vector index query tool. Even if the tool makes an error or only answers part of the question, the agent can continue retrying the question until the task is complete.

In [2]:
# Import standard libraries
from llama_index.core.agent import (
    CustomSimpleAgentWorker,
    Task,
    AgentChatResponse,
)
from typing import Dict, Any, List, Tuple, Optional
from llama_index.core.tools import BaseTool, QueryEngineTool
from llama_index.core.program import LLMTextCompletionProgram
from llama_index.core.output_parsers import PydanticOutputParser
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core import ChatPromptTemplate, PromptTemplate
from llama_index.core.selectors import PydanticSingleSelector
from llama_index.core.bridge.pydantic import Field, BaseModel, PrivateAttr
from llama_index.core.llms import ChatMessage, MessageRole

# Import RagaAI Catalyst components
from ragaai_catalyst.tracers import Tracer
from ragaai_catalyst import RagaAICatalyst, init_tracing
from ragaai_catalyst import trace_tool, trace_llm, trace_agent

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify environment variables
required_env_vars = [
    "RAGAAI_CATALYST_ACCESS_KEY",
    "RAGAAI_CATALYST_SECRET_KEY",
    "RAGAAI_CATALYST_BASE_URL",
    "OPENAI_API_KEY"
]

missing_vars = [var for var in required_env_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

# Initialize RagaAI Catalyst with error checking
try:
    catalyst = RagaAICatalyst(
        access_key=os.getenv("RAGAAI_CATALYST_ACCESS_KEY"),
        secret_key=os.getenv("RAGAAI_CATALYST_SECRET_KEY"),
        base_url=os.getenv("RAGAAI_CATALYST_BASE_URL"),
    )
    print("RagaAI Catalyst initialized successfully")
except Exception as e:
    raise Exception(f"Failed to initialize RagaAI Catalyst: {str(e)}")

# Initialize tracer with more specific naming
tracer = Tracer(
    project_name="LlamaIndex_Agent_Demo",
    dataset_name="custom_retry_agent",
    tracer_type="Agentic",
)

# Initialize tracing with error handling
try:
    init_tracing(catalyst=catalyst, tracer=tracer)
    print("Tracing initialized successfully")
except Exception as e:
    raise Exception(f"Error initializing tracing: {str(e)}")

DEFAULT_PROMPT_STR = """
Given previous question/response pairs, please determine if an error has occurred in the response, and suggest \
    a modified question that will not trigger the error.

Examples of modified questions:
- The question itself is modified to elicit a non-erroneous response
- The question is augmented with context that will help the downstream system better answer the question.
- The question is augmented with examples of negative responses, or other negative questions.

An error means that either an exception has triggered, or the response is completely irrelevant to the question.

Please return the evaluation of the response in the following JSON format.
"""

@trace_tool("get_chat_prompt_template")
def get_chat_prompt_template(
    system_prompt: str, current_reasoning: List[Tuple[str, str]]
) -> ChatPromptTemplate:
    system_msg = ChatMessage(role=MessageRole.SYSTEM, content=system_prompt)
    messages = [system_msg]
    for raw_msg in current_reasoning:
        if raw_msg[0] == "user":
            messages.append(
                ChatMessage(role=MessageRole.USER, content=raw_msg[1])
            )
        else:
            messages.append(
                ChatMessage(role=MessageRole.ASSISTANT, content=raw_msg[1])
            )
    return ChatPromptTemplate(message_templates=messages)

class ResponseEval(BaseModel):
    """Evaluation of whether the response has an error."""
    has_error: bool = Field(..., description="Whether the response has an error.")
    new_question: str = Field(..., description="The suggested new question.")
    explanation: str = Field(
        ...,
        description=(
            "The explanation for the error as well as for the new question."
            "Can include the direct stack trace as well."
        ),
    )

@trace_agent("RetryAgentWorker")
class RetryAgentWorker(CustomSimpleAgentWorker):
    """Agent worker that adds a retry layer on top of a router."""

    prompt_str: str = Field(default=DEFAULT_PROMPT_STR)
    max_iterations: int = Field(default=10)
    _router_query_engine: RouterQueryEngine = PrivateAttr()

    @classmethod
    def from_tools(
        cls,
        tools: List[BaseTool],
        llm: Any,
        **kwargs: Any,
    ) -> "RetryAgentWorker":
        """Create a RetryAgentWorker from a list of tools."""
        return cls(tools=tools, llm=llm, **kwargs)

    def __init__(self, tools: List[BaseTool], **kwargs: Any) -> None:
        for tool in tools:
            if not isinstance(tool, QueryEngineTool):
                raise ValueError(
                    f"Tool {tool.metadata.name} is not a query engine tool."
                )
        self._router_query_engine = RouterQueryEngine(
            selector=PydanticSingleSelector.from_defaults(),
            query_engine_tools=tools,
            verbose=kwargs.get("verbose", False),
        )
        super().__init__(tools=tools, **kwargs)

    def _initialize_state(self, task: Task, **kwargs: Any) -> Dict[str, Any]:
        return {"count": 0, "current_reasoning": []}

    @trace_llm("run_step")
    def _run_step(
        self, state: Dict[str, Any], task: Task, input: Optional[str] = None
    ) -> Tuple[AgentChatResponse, bool]:
        if "new_input" not in state:
            new_input = task.input
        else:
            new_input = state["new_input"]

        response = self._router_query_engine.query(new_input)

        state["current_reasoning"].extend(
            [("user", new_input), ("assistant", str(response))]
        )

        chat_prompt_tmpl = get_chat_prompt_template(
            self.prompt_str, state["current_reasoning"]
        )
        llm_program = LLMTextCompletionProgram.from_defaults(
            output_parser=PydanticOutputParser(output_cls=ResponseEval),
            prompt=chat_prompt_tmpl,
            llm=self.llm,
        )
        
        response_eval = llm_program(
            query_str=new_input, response_str=str(response)
        )
        if not response_eval.has_error:
            is_done = True
        else:
            is_done = False
        state["new_input"] = response_eval.new_question

        if self.verbose:
            print(f"> Question: {new_input}")
            print(f"> Response: {response}")
            print(f"> Response eval: {response_eval.dict()}")

        return AgentChatResponse(response=str(response)), is_done

    def _finalize_task(self, state: Dict[str, Any], **kwargs) -> None:
        pass

# Database setup
from sqlalchemy import create_engine, MetaData, Table, Column, String, Integer, insert
from llama_index.core import SQLDatabase

try:
    engine = create_engine("sqlite:///:memory:", future=True)
    metadata_obj = MetaData()

    city_stats_table = Table(
        "city_stats",
        metadata_obj,
        Column("city_name", String(16), primary_key=True),
        Column("population", Integer),
        Column("country", String(16), nullable=False),
    )

    metadata_obj.create_all(engine)

    # Insert data
    rows = [
        {"city_name": "Toronto", "population": 2930000, "country": "Canada"},
        {"city_name": "Tokyo", "population": 13960000, "country": "Japan"},
        {"city_name": "Berlin", "population": 3645000, "country": "Germany"},
    ]
    for row in rows:
        stmt = insert(city_stats_table).values(**row)
        with engine.begin() as connection:
            connection.execute(stmt)
    
    print("Database setup completed successfully")
except Exception as e:
    raise Exception(f"Database setup failed: {str(e)}")

# Setup query engines and tools
from llama_index.core.query_engine import NLSQLTableQueryEngine
from llama_index.readers.wikipedia import WikipediaReader
from llama_index.core import VectorStoreIndex

@trace_tool("setup_sql_tool")
def setup_sql_tool():
    try:
        sql_database = SQLDatabase(engine, include_tables=["city_stats"])
        sql_query_engine = NLSQLTableQueryEngine(
            sql_database=sql_database, tables=["city_stats"], verbose=True
        )
        return QueryEngineTool.from_defaults(
            query_engine=sql_query_engine,
            description=(
                "Useful for translating a natural language query into a SQL query over"
                " a table containing: city_stats, containing the population/country of"
                " each city"
            ),
        )
    except Exception as e:
        raise Exception(f"Failed to setup SQL tool: {str(e)}")

@trace_tool("setup_vector_tools")
def setup_vector_tools():
    try:
        cities = ["Toronto", "Berlin", "Tokyo"]
        wiki_docs = WikipediaReader().load_data(pages=cities)
        vector_tools = []
        for city, wiki_doc in zip(cities, wiki_docs):
            vector_index = VectorStoreIndex.from_documents([wiki_doc])
            vector_query_engine = vector_index.as_query_engine()
            vector_tool = QueryEngineTool.from_defaults(
                query_engine=vector_query_engine,
                description=f"Useful for answering semantic questions about {city}",
            )
            vector_tools.append(vector_tool)
        return vector_tools
    except Exception as e:
        raise Exception(f"Failed to setup vector tools: {str(e)}")

# Main execution
from llama_index.llms.openai import OpenAI

try:
    with tracer:
        # Initialize LLM
        llm = OpenAI(model="gpt-4")
        print("LLM initialized successfully")

        # Setup tools
        sql_tool = setup_sql_tool()
        vector_tools = setup_vector_tools()
        query_engine_tools = [sql_tool] + vector_tools
        print("Tools setup completed")

        # Initialize agent
        agent_worker = RetryAgentWorker(
            tools=query_engine_tools,
            llm=llm,
            verbose=True
        )
        agent = agent_worker.as_agent()
        print("Agent initialized successfully")
        
        # Add multiple complex queries
        queries = [
            "Compare the population density and cultural significance of Tokyo and Berlin. Include specific numbers and notable facts.",
            "For each city in the database, tell me its population and three major historical events. Sort the results by population in descending order.",
            "Which city has experienced the most significant population growth, and what major developments contributed to this growth? Include both statistical and historical context.",
            "Compare the transportation systems of all three cities and relate them to their respective population sizes. Include specific infrastructure details if available."
        ]
        
        print("\nExecuting multiple complex queries:")
        for i, query in enumerate(queries, 1):
            print(f"\n=== Query {i} ===")
            print(f"Question: {query}")
            response = agent.chat(query)
            print(f"Response: {str(response)}\n")
            print("-" * 80)
        
        print("\nTrace completed successfully")
except Exception as e:
    print(f"Error during execution: {str(e)}")
    raise

Token(s) set successfully
RagaAI Catalyst initialized successfully
Tracing initialized successfully
Database setup completed successfully
LLM initialized successfully


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Tools setup completed
Agent initialized successfully

Executing multiple complex queries:

=== Query 1 ===
Question: Compare the population density and cultural significance of Tokyo and Berlin. Include specific numbers and notable facts.


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:llama_index.core.query_engine.router_query_engine:Selecting query engine 3: Answering semantic questions about Tokyo.


Selecting query engine 3: Answering semantic questions about Tokyo.


...

> Question: Can you provide a comparison of the transportation systems of New York, London, and Tokyo, and relate them to their respective population sizes? Please include specific infrastructure details if available.
> Response: New York, London, and Tokyo are major global cities with diverse transportation systems. New York City has an extensive subway system operated by the Metropolitan Transportation Authority (MTA), serving a population of over 8 million in the city alone. London boasts the famous London Underground, managed by Transport for London (TfL), catering to a population of around 9 million in the city. Tokyo, on the other hand, has a highly developed rail network, including the JR East and Tokyo Metro lines, serving a population of over 14 million in the metropolitan area.

In terms of infrastructure, New York City's subway system is one of the oldest and busiest in the world, with over 400 stations and 24 subway lines. London's Underground is known for its iconic Tube n

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:llama_index.core.query_engine.router_query_engine:Selecting query engine 3: Answering semantic questions about Tokyo and comparing transportation systems of cities.


Selecting query engine 3: Answering semantic questions about Tokyo and comparing transportation systems of cities.


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:ragaai_catalyst.tracers.agentic_tracing.utils.zip_list_of_unique_files: Zip file created successfully.
INFO:ragaai_catalyst.tracers.agentic_tracing.tracers.base: Traces saved successfully.


> Question: Can you provide a comparison of the transportation systems of New York, London, and Tokyo, and relate them to their respective population sizes? Please include specific infrastructure details if available.
> Response: New York, London, and Tokyo are major global cities with diverse transportation systems. New York City has an extensive subway system operated by the Metropolitan Transportation Authority (MTA), with over 472 stations and 24 subway lines. London has the London Underground, commonly known as the Tube, which is one of the oldest and most extensive metro systems in the world, serving over 270 stations. Tokyo, on the other hand, boasts the most extensive urban railway network globally, with up to 62 electric train lines and more than 900 train stations.

In terms of population size, New York City is the most populous of the three, with over 8 million residents in the city proper and over 20 million in the metropolitan area. London follows with a city population of